# Chapter 3: five tables and the orphans
The week comes as five CSV files: sample, test, result, spec_limit and disposition. Every result should point at a real sample, a real test and a spec in force. This notebook lists the rows that do not.

**No code tier.** Put the five files on five tabs. In the result tab add three lookup columns: does the sample ID exist, does the test code exist, is there a spec for this beer and test. Filter for the blanks.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"

In [ ]:
sample = pd.read_csv(DATA + "week_sample.csv", keep_default_na=False)
test = pd.read_csv(DATA + "week_test.csv")
result = pd.read_csv(DATA + "week_result.csv", keep_default_na=False)
spec = pd.read_csv(DATA + "week_spec_limit.csv")
disposition = pd.read_csv(DATA + "week_disposition.csv")
for name, df in (("sample", sample), ("test", test), ("result", result), ("spec_limit", spec), ("disposition", disposition)):
    print(f"{name:12s}{len(df):6d} rows")

In [ ]:
no_sample = result[~result.sample_id.isin(sample.sample_id)]
no_test = result[~result.test_code.isin(test.test_code)]
print("results with no sample:"); print(no_sample[["result_id", "sample_id", "test_code"]])
print("results with no test:"); print(no_test[["result_id", "sample_id", "test_code"]])

In [ ]:
joined = result.merge(sample[["sample_id", "beer", "sampled_at"]], on="sample_id", how="inner")
joined = joined[joined.beer != ""]
keys = set(zip(spec.beer, spec.test_code))
has_spec = [(b, t) in keys or ("all", t) in keys for b, t in zip(joined.beer, joined.test_code)]
no_spec = joined[~np.array(has_spec)]
print(len(no_spec), "beer results with no spec in force:")
print(no_spec.groupby(["beer", "test_code"]).size())

In [ ]:
key = sample[["fill_id", "brew_id", "lot_id", "equipment_id"]].ne("").any(axis=1)
print("share of samples with a batch or equipment key:", round(100 * key.mean(), 1), "percent")
print(sample.link_method.value_counts())
s = result[result.sample_id == "S261031-017"]
print(s[["test_code", "value", "analyst", "tested_at", "entered_at", "approved_at"]])

**What to look at.** The orphan lists: two typed sample IDs, a test code with no angle (HAZE) and every pale ale CO2 result, because the spec table has no pale ale CO2 row. Then the four timestamps of S261031-017: 4 hours 15 minutes from pull to approval.